# NOORIX-AI PKR Currency Classification — MobileNetV2
**Prepared by Muhammad Farhan | NOORIX-AI**

Classifies visible Pakistani banknote denomination/side labels only. It is NOT counterfeit detection or authentication.

## Kaggle notebook title
`NOORIX-PKR-Currency-MobileNetV2-Muhammad-Farhan`

## Dataset to attach
Search Kaggle: `Pakistanicurrencydataset` (owner `muhammadammarabid`) or use the PKR dataset already downloaded. Dataset must be ImageFolder-style: one folder per class, images inside each class folder. Do not mix Afghani and PKR classes. Generic `Back`/`Front` folders are excluded.

## Output
`NOORIX_PKR_MobileNetV2_Muhammad_Farhan.zip`


In [ ]:
!pip -q install -U onnx onnxscript scikit-learn
import os, glob, json, shutil, random, sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Python:',sys.version); print('PyTorch:',torch.__version__); print('Device:',device)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
ROOTS=[Path(p) for p in glob.glob('/kaggle/input/*')]
print('Kaggle inputs:'); [print(' -',p) for p in ROOTS]


In [ ]:
# Set this to the directory whose immediate subfolders are class names.
DATA_DIR=''
if not DATA_DIR: raise ValueError('Set DATA_DIR to the ImageFolder root shown under /kaggle/input.')
DATA_DIR=Path(DATA_DIR)
if not DATA_DIR.is_dir(): raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}')
IMG_SIZE=224; BATCH_SIZE=16; EPOCHS=6
transform=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
full=datasets.ImageFolder(str(DATA_DIR),transform=transform)
excluded={'back','front','reverse','obverse','background','unknown'}
labels=[name for name in full.classes if name.strip().lower() not in excluded]
if len(labels)<2: raise ValueError(f'Need at least 2 denomination classes after filtering. Found: {labels}')
old_to_new={full.class_to_idx[name]:i for i,name in enumerate(labels)}
indices=[i for i,target in enumerate(full.targets) if target in old_to_new]
targets=[old_to_new[full.targets[i]] for i in indices]
counts={name:sum(1 for y in targets if y==i) for i,name in enumerate(labels)}
print('Classes:',labels); print('Class counts:',counts); print('Usable images:',len(indices))
if len(indices)<20 or min(counts.values())<5: raise ValueError('Need at least 20 usable images and at least 5 images per class for a first training run.')
try: tr_idx,va_idx=train_test_split(np.arange(len(indices)),test_size=0.2,random_state=SEED,stratify=targets)
except ValueError as e: raise ValueError(f'Cannot create a stratified train/validation split: {e}')
class RemappedSubset(torch.utils.data.Dataset):
    def __init__(self,base,positions): self.base=base; self.positions=list(positions)
    def __len__(self): return len(self.positions)
    def __getitem__(self,idx):
        original_index=indices[self.positions[idx]]
        image,old_label=self.base[original_index]
        return image,old_to_new[old_label]
train_ds=RemappedSubset(full,tr_idx); val_ds=RemappedSubset(full,va_idx)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(device=='cuda'))
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=(device=='cuda'))
print('Train:',len(train_ds),'Validation:',len(val_ds))


In [ ]:
weights=models.MobileNet_V2_Weights.DEFAULT
model=models.mobilenet_v2(weights=weights)
model.classifier[1]=nn.Linear(model.last_channel,len(labels))
model=model.to(device)
criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4)
best_acc=-1.0; best_path=Path('/kaggle/working/mobilenetv2_pkr_best.pth')
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for images,y in train_loader:
        images=images.to(device); y=y.to(device)
        optimizer.zero_grad(set_to_none=True); logits=model(images); loss=criterion(logits,y); loss.backward(); optimizer.step(); losses.append(loss.item())
    model.eval(); truth=[]; pred=[]
    with torch.no_grad():
        for images,y in val_loader:
            logits=model(images.to(device)); pred.extend(logits.argmax(1).cpu().tolist()); truth.extend(y.tolist())
    acc=accuracy_score(truth,pred)
    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss={np.mean(losses):.4f} | val_accuracy={acc:.4f}')
    if acc>best_acc: best_acc=acc; torch.save(model.state_dict(),best_path)
if not best_path.exists(): raise RuntimeError('Training did not produce a checkpoint.')
print('Best validation accuracy:',best_acc)
print(classification_report(truth,pred,labels=list(range(len(labels))),target_names=labels,zero_division=0))


In [ ]:
model.load_state_dict(torch.load(best_path,map_location=device,weights_only=True)); model.eval()
out=Path('/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan'); out.mkdir(parents=True,exist_ok=True)
torch.save(model.cpu().state_dict(),out/'mobilenetv2_pkr_best.pth')
(out/'labels.json').write_text(json.dumps({'classes':labels},ensure_ascii=False,indent=2),encoding='utf-8')
metadata={'project':'NOORIX-AI','prepared_by':'Muhammad Farhan','architecture':'MobileNetV2','input_size':[1,3,224,224],'normalization':{'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225]},'validation_accuracy':float(best_acc),'train_samples':len(train_ds),'validation_samples':len(val_ds),'warning':'Denomination/side classification only; not counterfeit detection.'}
(out/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
dummy=torch.zeros(1,3,224,224)
torch.onnx.export(model,dummy,str(out/'mobilenetv2_pkr.onnx'),input_names=['image'],output_names=['logits'],dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}},opset_version=17,dynamo=False)
zip_path=shutil.make_archive('/kaggle/working/NOORIX_PKR_MobileNetV2_Muhammad_Farhan','zip',out)
print('DOWNLOAD THIS FILE:',zip_path); print('ZIP contents:'); [print(' -',p.name) for p in out.iterdir()]
